# Общий набор Г1: NMT + турецкий набор, один раз для всех блокнотов моделей

| Поле | Значение |
|------|----------|
| **Автор** | Захар Санников (zs) |
| **Дата** | 2026-10-26 |
| **Гипотеза** | Г1 (ЭЭГ) |
| **Зачем** | Каждый блокнот дообучения заново читал и фильтровал NMT (2 417 записей) и турецкий набор — 30–75 мин на сеанс. Здесь это делается один раз; выход блокнота подключается как источник |
| **Среда** | Kaggle, сеанс **без GPU** (`zakharsann/h1-prep-shared`) |
| **Источники** | `mhmdrdwn/nmt-eeg`, `buraktaci/turkish-epilepsy` |
| **Подключение** | в `kernel-metadata.json` блокнота модели: `"kernel_sources": ["zakharsann/h1-prep-shared"]`; загрузчик — `h1_eeg/src/shared_data.py` (скопировать в первую ячейку) |

**Чтение — как в блокнотах EEG-DINO (2026-10-22) и REVE v2 (2026-10-23).**
- NMT: `mne.io.read_raw_edf`, частота 200 Гц (в файлах уже 200 Гц; если нет — `raw.resample(200)`), 19 каналов по именам в порядке EL, без перереференции: в файлах NMT уже средний референт (так же было в старых блокнотах).
- Турция: первые 19 каналов (относительно уха) в порядке EL → средний референт по 19 каналам → 500→200 Гц (`resample_poly(2, 5)`) по каждому куску 15 с → куски подряд.
- Порядок каналов EL: Fp1 Fp2 F3 F4 C3 C4 P3 P4 O1 O2 F7 F8 T3 T4 T5 T6 Fz Cz Pz.
- Фильтр: Баттерворт 4-го порядка 0,1–75 Гц (`sosfiltfilt`) + режекция 50 Гц (`iirnotch`, Q = 30, `filtfilt`) по записи целиком. Это общая нижняя граница; блокнот модели может дофильтровать (например, 0,3 Гц — `shared_data.highpass`). Отличие от старого кода: там записи короче 60 с не фильтровались, здесь фильтруются все.
- **Масштаб NMT**: после MNE СКО ≈ 0,2 мкВ (масштаб в файлах неверный). Сигнал умножается на общий множитель `K` (в `h1_shared_info.json` и в столбце `scale_k`): `K = 10^round(lg(медиана СКО здоровых Турции / медиана СКО первых 150 «норм» NMT train))`. Нормировки окна нет — её делает блокнот модели.
- Хранение: float16, мкВ; значения обрезаются до ±60 000 (число обрезанных отсчётов — столбец `clipped`).

**Формат выхода** (`/kaggle/input/<источник>/…`, найти папку — `shared_data.find_root()`):
- `h1_shared_meta.csv` — строка на запись: `dataset` (nmt / turkey), `part` (train / eval / test), `split` (fit / val / eval / turkey), `id` (recordname NMT или имя .mat), `y` (1 — патология NMT / эпилепсия), `label`, `sfreq_orig`, `dur_full_s` (длительность исходной записи), `dur_s` (сохранено), `scale_k`, `std_uv` (СКО отфильтрованной записи целиком, мкВ — для проверки связи с амплитудой), `std_raw_uv` (СКО до фильтра; у Турции = `TR_AMP` старых блокнотов), `clipped`, `n_segments` (Турция), `shard200`/`off200`/`n200` (и `…256`, если сохранён вариант 256 Гц).
- `fs200/<часть>_<номер>.npy` — плоские массивы float16 по ~1 ГБ; запись = `arr[off : off + 19·n].reshape(19, n)`.
- `h1_shared_info.json` — параметры подготовки, множитель, решение по объёму, времена.
- `h1_shared_failed.csv` — записи NMT, которые не прочитались (как в старых блокнотах: 3 записи train).

**Деление NMT train** (столбец `split`): `train_test_split(recordname, test_size=0.15, stratify=y, random_state=0)` по прочитанным записям train, отсортированным по имени, — ровно как в блокнотах EEG-DINO / REVE / BIOT / LUNA. `val` — отложенные 15 % (выбор настроек, эпохи и порога).

**Объём**: оценивается по заголовкам EDF до чтения (ячейка 3). Лимит выхода Kaggle ~20 ГБ, бюджет здесь 18 ГБ. Если всё не помещается — для NMT train хранятся первые N минут каждой записи (N — наибольшее целое ≥ 5, при котором помещается), NMT eval и Турция — целиком. Вариант 256 Гц — только если помещается вместе с 200 Гц. Итог решения — в `h1_shared_info.json` и в выводе ниже.

In [ ]:
import os, glob, time, json, warnings
import numpy as np, pandas as pd, scipy.io, scipy.signal as ss
from fractions import Fraction
from concurrent.futures import ThreadPoolExecutor
from sklearn.model_selection import train_test_split
import mne
mne.set_log_level("ERROR")
warnings.filterwarnings("ignore", category=RuntimeWarning)
T_NB = time.time(); TIMES = {}
OUT = "/kaggle/working"

SFREQ, FS_EXTRA, L10 = 200, 256, 2000
EL = ["Fp1", "Fp2", "F3", "F4", "C3", "C4", "P3", "P4", "O1", "O2", "F7", "F8", "T3", "T4", "T5", "T6", "Fz", "Cz", "Pz"]
BAND, NOTCH = (0.1, 75.0), 50.0
BUDGET, SHARD_BYTES, F16_MAX, MIN_TRAIN_MIN, K_EST_N = 18.0e9, 1.0e9, 60000.0, 5, 150

SOS_BP = ss.butter(4, list(BAND), btype="bandpass", fs=SFREQ, output="sos")
B_N, A_N = ss.iirnotch(NOTCH, 30, fs=SFREQ)
def filt(X):
    """0,1-75 Гц + режекция 50 Гц: тот же код, что cb_filter старых блокнотов, но нижняя граница 0,1 и без пропуска записей < 60 с."""
    return ss.filtfilt(B_N, A_N, ss.sosfiltfilt(SOS_BP, X.astype(np.float64), axis=-1), axis=-1)

FR = Fraction(FS_EXTRA, SFREQ)
def to256(X):
    return ss.resample_poly(X, FR.numerator, FR.denominator, axis=-1)

def to16(X):
    n_clip = int((np.abs(X) > F16_MAX).sum())
    return np.clip(X, -F16_MAX, F16_MAX).astype(np.float16), n_clip

ROOT_NMT = glob.glob("/kaggle/input/**/nmt_scalp_eeg_dataset", recursive=True)[0]
ROOT_TR = glob.glob("/kaggle/input/**/Turkish Epilepsy Dataset", recursive=True)[0]
lab = pd.read_csv(f"{ROOT_NMT}/Labels.csv")
paths = {os.path.basename(p): p for p in glob.glob(f"{ROOT_NMT}/**/*.edf", recursive=True)}
lab["path"] = lab["recordname"].map(paths)
lab = lab.dropna(subset=["path"]).sort_values("recordname").reset_index(drop=True)
lab["y"] = (lab["label"].str.lower() == "abnormal").astype(int)
tfiles = [(f, 1) for f in sorted(glob.glob(f"{ROOT_TR}/epilepsy/*.mat"))] + \
         [(f, 0) for f in sorted(glob.glob(f"{ROOT_TR}/normal/*.mat"))]
print(pd.crosstab(lab["loc"], lab["label"])); print("Турция:", len(tfiles), "файлов")

def warm(p):
    with open(p, "rb") as f:
        while f.read(1 << 24):
            pass
t0 = time.time(); allf = list(lab.path) + [f for f, _ in tfiles]
with ThreadPoolExecutor(32) as ex:
    for i, _ in enumerate(ex.map(warm, allf)):
        if (i + 1) % 500 == 0 or i + 1 == len(allf):
            print(f"подкачано {i+1}/{len(allf)} {time.time()-t0:.0f} с", flush=True)
TIMES["подкачка"] = time.time() - t0

ALIAS = {"T7": "T3", "T8": "T4", "P7": "T5", "P8": "T6"}
TARGET = [e.upper() for e in EL]
def norm(ch):
    c = ch.upper().replace("EEG", "").strip()
    for s in ["-REF", "-AVG", "-LE", "-AR"]:
        c = c.replace(s, "")
    return ALIAS.get(c.strip(" -_"), c.strip(" -_"))
# индексы турецких каналов (первые 19, относительно уха) в порядке EL
IDX = [0, 1, 2, 3, 5, 6, 8, 9, 11, 12, 13, 14, 15, 16, 17, 18, 4, 7, 10]

## 2. Заголовки NMT (длительность, частота) и турецкий набор целиком

In [ ]:
def hdr(p):
    try:
        raw = mne.io.read_raw_edf(p, preload=False)
        return float(raw.info["sfreq"]), int(raw.n_times), "|".join(raw.ch_names)
    except Exception:
        return np.nan, 0, ""
t0 = time.time()
with ThreadPoolExecutor(16) as ex:
    H = list(ex.map(hdr, lab.path))
lab["sfreq_orig"] = [h[0] for h in H]; lab["n_orig"] = [h[1] for h in H]
lab["dur_full_s"] = lab.n_orig / lab.sfreq_orig
TIMES["заголовки NMT"] = time.time() - t0
print(f"заголовки за {TIMES['заголовки NMT']:.0f} с | частоты:", lab.sfreq_orig.value_counts(dropna=False).to_dict())
print("наборы каналов (первые 3 варианта):", pd.Series([h[2] for h in H]).value_counts().head(3).to_dict())
print("длительность NMT, мин:"); print((lab.groupby("loc").dur_full_s.describe().drop(columns="count") / 60).round(1))

def tr_job(f):
    d = scipy.io.loadmat(f)["data"]
    X = np.stack([np.stack([d[s, c].ravel() for c in IDX]) for s in range(d.shape[0])])   # (куски, 19, 7500)
    X = X - X.mean(1, keepdims=True)          # средний референт
    X = ss.resample_poly(X, 2, 5, axis=-1)    # 200 Гц
    std_raw = float(np.std(X))
    Xf = filt(np.concatenate(list(X), axis=-1))
    a16, clip = to16(Xf)
    return dict(n_segments=int(d.shape[0]), std_raw_uv=std_raw, std_uv=float(np.std(Xf)), clipped=clip), a16, to16(to256(Xf))[0]

t0 = time.time()
with ThreadPoolExecutor(4) as ex:
    TRK = list(ex.map(tr_job, [f for f, _ in tfiles]))
TIMES["Турция"] = time.time() - t0
tr_n = np.array([a.shape[1] for _, a, _ in TRK])
print(f"Турция прочитана за {TIMES['Турция']:.0f} с | длительность, мин: медиана {np.median(tr_n)/SFREQ/60:.1f}, мин {tr_n.min()/SFREQ/60:.1f}, макс {tr_n.max()/SFREQ/60:.1f}")

## 3. Множитель масштаба NMT и решение по объёму

In [ ]:
def std_job(p):
    try:
        raw = mne.io.read_raw_edf(p, preload=True)
        if raw.info["sfreq"] != SFREQ:
            raw.resample(SFREQ)
        names = {norm(c): c for c in raw.ch_names}
        return float(np.std(filt(raw.get_data(picks=[names[t] for t in TARGET]) * 1e6)))
    except Exception:
        return np.nan
t0 = time.time()
est = lab[(lab["loc"] == "train") & (lab.y == 0)].head(K_EST_N)
with ThreadPoolExecutor(4) as ex:
    s_nmt = np.array(list(ex.map(std_job, est.path)))
med_nmt = float(np.nanmedian(s_nmt))
med_tr = float(np.median([m["std_uv"] for (m, _, _), (_, yy) in zip(TRK, tfiles) if yy == 0]))
K = float(10 ** round(np.log10(med_tr / med_nmt)))
TIMES["множитель"] = time.time() - t0
print(f"СКО после MNE·1e6: NMT «норма» (первые {len(est)} train) медиана {med_nmt:.3f} мкВ | здоровые Турции {med_tr:.2f} мкВ | отношение {med_tr/med_nmt:.1f} → K = {K:g}")
print(f"NMT после K: медиана СКО {med_nmt*K:.1f} мкВ")

n200 = np.round(lab.n_orig * SFREQ / lab.sfreq_orig.fillna(SFREQ)).astype(np.int64).to_numpy()
is_tr = (lab["loc"] == "train").to_numpy()
def size200(cap_min=None):
    n = n200.copy()
    if cap_min is not None:
        n[is_tr] = np.minimum(n[is_tr], cap_min * 60 * SFREQ)
    return 19 * 2 * (n.sum() + tr_n.sum())
full200 = size200(); full256 = full200 * FS_EXTRA / SFREQ
if full200 + full256 <= BUDGET:
    SAVE256, CAP_MIN = True, None
elif full200 <= BUDGET:
    SAVE256, CAP_MIN = False, None
else:
    SAVE256 = False
    CAP_MIN = next((m for m in range(int(n200[is_tr].max() // (60 * SFREQ)) + 1, 0, -1) if size200(m) <= BUDGET), 0)
    assert CAP_MIN >= MIN_TRAIN_MIN, f"даже {MIN_TRAIN_MIN} мин NMT train не помещаются"
DECISION = (f"всё 200 Гц {full200/1e9:.1f} ГБ, вместе с 256 Гц {(full200+full256)/1e9:.1f} ГБ, бюджет {BUDGET/1e9:.0f} ГБ → "
            + ("сохраняю целиком 200 и 256 Гц" if SAVE256 else
               "сохраняю целиком 200 Гц, без 256 Гц" if CAP_MIN is None else
               f"200 Гц: NMT train — первые {CAP_MIN} мин записи, NMT eval и Турция целиком ({size200(CAP_MIN)/1e9:.1f} ГБ); 256 Гц не сохраняю"))
print(DECISION)
if CAP_MIN is not None:
    print(f"записей NMT train длиннее {CAP_MIN} мин: {(n200[is_tr] > CAP_MIN*60*SFREQ).mean():.0%}")

## 4. Запись: Турция, затем NMT (параллельно 4 потока, куски по ~1 ГБ)

In [ ]:
class Shards:
    """Плоские float16-массивы по ~1 ГБ: записи (19, n) подряд."""
    def __init__(self, fs, part):
        self.dir, self.part, self.k, self.buf, self.n_el = f"{OUT}/fs{fs}", part, 0, [], 0
        os.makedirs(self.dir, exist_ok=True)
    def add(self, X16):
        if self.buf and (self.n_el + X16.size) * 2 > SHARD_BYTES:
            self.flush()
        off = self.n_el
        self.buf.append(X16.ravel()); self.n_el += X16.size
        return f"{self.part}_{self.k:03d}.npy", off, X16.shape[1]
    def flush(self):
        if self.buf:
            np.save(f"{self.dir}/{self.part}_{self.k:03d}.npy", np.concatenate(self.buf))
            self.k, self.buf, self.n_el = self.k + 1, [], 0

FSS = [SFREQ] + ([FS_EXTRA] if SAVE256 else [])
ROWS = []
def put(row, xs, writers):
    for fs, X16 in zip(FSS, xs):
        row[f"shard{fs}"], row[f"off{fs}"], row[f"n{fs}"] = writers[fs].add(X16)
    ROWS.append(row)

t0 = time.time()
W = {fs: Shards(fs, "turkey") for fs in FSS}
for (f, yy), (m, a16, b16) in zip(tfiles, TRK):
    put(dict(dataset="turkey", part="test", id=os.path.basename(f), y=yy, label="epilepsy" if yy else "normal", sfreq_orig=500.0,
             dur_full_s=a16.shape[1] / SFREQ, dur_s=a16.shape[1] / SFREQ, scale_k=1.0, **m), [a16, b16], W)
for w in W.values(): w.flush()
del TRK

def nmt_job(i):
    r = lab.iloc[i]
    try:
        raw = mne.io.read_raw_edf(r.path, preload=True)
        if raw.info["sfreq"] != SFREQ:
            raw.resample(SFREQ)
        names = {norm(c): c for c in raw.ch_names}
        X = raw.get_data(picks=[names[t] for t in TARGET]) * 1e6 * K
    except Exception as e:
        return i, None, f"чтение: {type(e).__name__}: {str(e)[:150]}"
    if X.shape[1] < L10:
        return i, None, f"короче 10 с ({X.shape[1]} отсчётов)"
    Xf = filt(X)
    n_keep = Xf.shape[1] if (r["loc"] != "train" or CAP_MIN is None) else min(Xf.shape[1], CAP_MIN * 60 * SFREQ)
    Xs = Xf[:, :n_keep]
    a16, clip = to16(Xs)
    xs = [a16] + ([to16(to256(Xs))[0]] if SAVE256 else [])
    m = dict(dataset="nmt", part=r["loc"], id=r.recordname, y=int(r.y), label=r.label, sfreq_orig=r.sfreq_orig,
             dur_full_s=X.shape[1] / SFREQ, dur_s=n_keep / SFREQ, scale_k=K, std_uv=float(np.std(Xf)),
             std_raw_uv=float(np.std(X)), clipped=clip, short_lt60s=bool(X.shape[1] < 60 * SFREQ))
    return i, (m, xs), None

FAILED = []
for part in ["train", "eval"]:
    W = {fs: Shards(fs, f"nmt_{part}") for fs in FSS}
    idx = np.where(lab["loc"] == part)[0]
    with ThreadPoolExecutor(4) as ex:
        for j, (i, res, err) in enumerate(ex.map(nmt_job, idx)):
            if res is None:
                FAILED.append(dict(id=lab.recordname[i], part=part, error=err))
            else:
                put(res[0], res[1], W)
            if (j + 1) % 250 == 0 or j + 1 == len(idx):
                print(f"NMT {part}: {j+1}/{len(idx)} {time.time()-t0:.0f} с", flush=True)
    for w in W.values(): w.flush()
TIMES["чтение, фильтр и запись"] = time.time() - t0
print("не прочитались:", FAILED)

## 5. Деление NMT train (как в старых блокнотах), таблица метаданных, итог

In [ ]:
meta = pd.DataFrame(ROWS)
meta["split"] = np.where(meta.dataset == "turkey", "turkey", np.where(meta.part == "eval", "eval", "fit"))
tr = meta[(meta.dataset == "nmt") & (meta.part == "train")].sort_values("id").reset_index(drop=True)
fit_names, val_names = train_test_split(tr["id"].to_numpy(), test_size=0.15, stratify=tr.y, random_state=0)
meta.loc[meta["id"].isin(set(val_names)) & (meta.dataset == "nmt"), "split"] = "val"
first = ["dataset", "part", "split", "id", "y", "label", "sfreq_orig", "dur_full_s", "dur_s", "scale_k", "std_uv", "std_raw_uv", "clipped"]
meta = meta[first + [c for c in meta.columns if c not in first]]
meta.to_csv(f"{OUT}/h1_shared_meta.csv", index=False)
pd.DataFrame(FAILED, columns=["id", "part", "error"]).to_csv(f"{OUT}/h1_shared_failed.csv", index=False)

files = sorted(glob.glob(f"{OUT}/fs*/*.npy"))
total = sum(os.path.getsize(f) for f in files)
TIMES["всего"] = time.time() - T_NB
info = dict(version=1, created="2026-10-26", notebook="zakharsann/h1-prep-shared", sources=["mhmdrdwn/nmt-eeg", "buraktaci/turkish-epilepsy"],
            channels=EL, reference="NMT: средний референт из файлов (без перереференции); Турция: первые 19 каналов (ухо) → средний по 19",
            fs=FSS, filter=dict(band_hz=list(BAND), butter_order=4, sosfiltfilt=True, notch_hz=NOTCH, notch_q=30), dtype="float16", unit="мкВ",
            f16_clip=F16_MAX, nmt_scale_k=K, nmt_scale_evidence=dict(nmt_normal_std_after_mne_uv=med_nmt, turkey_normal_std_uv=med_tr, n_nmt=int(len(est))),
            nmt_train_cap_min=CAP_MIN, decision=DECISION, budget_gb=BUDGET / 1e9, size_gb=round(total / 1e9, 2), n_files=len(files),
            split="NMT train (прочитанные, по имени) → train_test_split(test_size=0.15, stratify=y, random_state=0); val = отложенные 15 %",
            times_s={k: round(v, 1) for k, v in TIMES.items()})
json.dump(info, open(f"{OUT}/h1_shared_info.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)

print(pd.crosstab([meta.dataset, meta.split], meta.y))
print(f"файлов {len(files)}, объём {total/1e9:.2f} ГБ | часы сигнала: " +
      ", ".join(f"{k} {v/3600:.0f}" for k, v in meta.groupby(["dataset", "part"]).dur_s.sum().items()))
print("СКО записи (мкВ), медиана:", meta.groupby(["dataset", "y"]).std_uv.median().round(1).to_dict(), "| обрезано отсчётов:", int(meta.clipped.sum()))
print("время, мин:", {k: round(v / 60, 1) for k, v in TIMES.items()})

## 6. Вывод Kaggle (версия 1, без строк о подкачке и служебных предупреждений)

Сеанс без GPU, всё заняло **5,2 мин**. В этом прогоне таблица длительностей печаталась с ошибкой в строке `count` (делилась на 60: 3,1 = 185 записей, 37,2 = 2 232); в коде выше это исправлено, на данные не влияет. Не прочитались те же 3 записи NMT train, что и в старых блокнотах (короче 10 с). Деление сверено с `results/eegdino_finetune_per_person.csv`: отложенные 335, NMT eval 185 и Турция 121 — те же id.

```
label  abnormal  normal
loc                    
eval         90      95
train       325    1907
Турция: 121 файлов
заголовки за 10 с | частоты: {200.0: 2417}
наборы каналов (первые 3 варианта): {'FP1|FP2|F3|F4|C3|C4|P3|P4|O1|O2|F7|F8|T3|T4|T5|T6|FZ|PZ|CZ|A1|A2': 2417}
длительность NMT, мин:
       count  mean  std  min   25%   50%   75%    max
loc                                                  
eval     3.1  11.9  3.9  0.3   9.8  11.5  14.0   27.6
train   37.2  12.2  5.0  0.1  10.0  11.7  14.2  112.5
Турция прочитана за 76 с | длительность, мин: медиана 18.5, мин 8.5, макс 67.0
СКО после MNE·1e6: NMT «норма» (первые 150 train) медиана 0.169 мкВ | здоровые Турции 12.75 мкВ | отношение 75.3 → K = 100
NMT после K: медиана СКО 16.9 мкВ
всё 200 Гц 14.6 ГБ, вместе с 256 Гц 33.2 ГБ, бюджет 18 ГБ → сохраняю целиком 200 Гц, без 256 Гц
NMT train: 250/2232 16 с
NMT train: 500/2232 34 с
NMT train: 750/2232 51 с
NMT train: 1000/2232 67 с
NMT train: 1250/2232 84 с
NMT train: 1500/2232 103 с
NMT train: 1750/2232 119 с
NMT train: 2000/2232 139 с
NMT train: 2232/2232 160 с
NMT eval: 185/185 174 с
не прочитались: [{'id': '0000654.edf', 'part': 'train', 'error': 'короче 10 с (738 отсчётов)'}, {'id': '0001596.edf', 'part': 'train', 'error': 'короче 10 с (1800 отсчётов)'}, {'id': '0001858.edf', 'part': 'train', 'error': 'короче 10 с (923 отсчётов)'}]
y                  0    1
dataset split            
nmt     eval      95   90
        fit     1618  276
        val      286   49
turkey  turkey    71   50
файлов 17, объём 14.56 ГБ | часы сигнала: ('nmt', 'eval') 37, ('nmt', 'train') 452, ('turkey', 'test') 43
СКО записи (мкВ), медиана: {('nmt', 0): 18.5, ('nmt', 1): 31.6, ('turkey', 0): 12.7, ('turkey', 1): 27.2} | обрезано отсчётов: 0
время, мин: {'подкачка': 0.7, 'заголовки NMT': 0.2, 'Турция': 1.3, 'множитель': 0.2, 'чтение, фильтр и запись': 2.9, 'всего': 5.2}
```

## 7. Проверка — отдельный блокнот `zakharsann/h1-prep-shared-check` (`h1_2026-10-26_zs_prep-shared-check.ipynb`)

Блокнот подключает этот выход через `kernel_sources` — так же, как блокноты моделей.

- **Время чтения** всего набора через `shared_data.read_all`: **53 с** (14,6 ГБ). Первая версия загрузчика читала записи срезами через `mmap`, и на `/kaggle/input` это заняло 2,1 ч. Поэтому `read_all` читает каждый файл-кусок целиком, а `mmap` (`read_record`) оставлен только для единичных записей. Время NMT eval (1,2 с) занижено: этот кусок перед этим уже читался тестом последовательного чтения.
- **Совпадение (а)** — старый код EEG-DINO с параметрами общего набора (0,1 Гц, ×100). На 3 записях NMT (train «норма», train «патология», eval) и 3 турецких (случайный выбор, все три оказались здоровыми) окна совпали **бит в бит**: доля равных значений 1,0, расхождение 0 мкВ.
- **(б)** — старый код как был (0,3–75 Гц) на окнах, которые брал старый блокнот, против общего набора с дофильтровкой `highpass(0,3)`. Медиана относительной ошибки 0,04–0,06 %. Максимум (5–25 %) приходится только на первое или последнее окно записи — это краевой эффект `filtfilt`. У записей NMT train, где старый блокнот не брал крайних окон, максимум 0,04 %.

```
источник: /kaggle/input/notebooks/zakharsann/h1-prep-shared
{'fs': [200], 'nmt_scale_k': 100.0, 'nmt_train_cap_min': None, 'decision': 'всё 200 Гц 14.6 ГБ, вместе с 256 Гц 33.2 ГБ, бюджет 18 ГБ → сохраняю целиком 200 Гц, без 256 Гц', 'size_gb': 14.56}
последовательное чтение одного куска nmt_eval_000.npy: 1.00 ГБ за 6.4 с
{'fs': 200, 'part': 'nmt/eval', 'records': 185, 'gb': 1.0, 'sec': 1.2}
{'fs': 200, 'part': 'nmt/train', 'records': 2229, 'gb': 12.38, 'sec': 41.5}
{'fs': 200, 'part': 'turkey/test', 'records': 121, 'gb': 1.18, 'sec': 10.0}
всего: 14.6 ГБ за 53 с (0.28 ГБ/с)
dataset  part                id  y  windows_old  windows_shared  compared  exact_equal  max_abs_uv  max_ulp_f16  old03_windows  old03_rel_rms_median  old03_rel_rms_max  old03_argmax_window  last_window
    nmt train       0001275.edf  0           60              60        60          1.0         0.0          0.0              8               0.00036            0.00037                   11           59
    nmt train       0001125.edf  1           65              65        65          1.0         0.0          0.0              8               0.00036            0.00036                   26           64
    nmt  eval       0000204.edf  0           70              70        70          1.0         0.0          0.0             60               0.00036            0.04754                    0           69
 turkey  test 2_Normal (51).mat  0          102             102       102          1.0         0.0          0.0             60               0.00043            0.19510                  101          101
 turkey  test 2_Normal (13).mat  0          132             132       132          1.0         0.0          0.0             60               0.00042            0.25008                  131          131
 turkey  test 2_Normal (18).mat  0           91              91        91          1.0         0.0          0.0             60               0.00060            0.12457                    0           90
совпадение (а): ДА | всё время блокнота 1.7 мин
```